First, Need to rotate both images so that they are oriented the same way 

In [1]:
pip install opencv-python

Note: you may need to restart the kernel to use updated packages.


In [2]:
import cv2
import numpy as np
import pandas as pd

In [3]:
# Load images
sat_image = cv2.imread('C:\\Users\melis\\OneDrive\\Documents\\School\\Capstone\\huron_county_satelite_map.png', cv2.IMREAD_COLOR)
ref_image = cv2.imread('C:\\Users\\melis\\OneDrive\\Documents\\School\\Capstone\\huron_county_full_vector_map.png', cv2.IMREAD_COLOR)

In [4]:
window_name = 'sat_image'
cv2.imshow(window_name, sat_image)

In [15]:
window_name = 'sat_image'
cv2.imshow(window_name, ref_image)

First, Need to rotate both images so that they are oriented the same way 

In [9]:
#function to rotate images
def rotate_image(image, angle):
    """Rotate an image by a given angle while keeping size."""
    (h, w) = image.shape[:2]
    center = (w // 2, h // 2)
    rotation_matrix = cv2.getRotationMatrix2D(center, angle, 1.0)
    rotated = cv2.warpAffine(image, rotation_matrix, (w, h))
    return rotated

In [ ]:
# Rotate first image
rotation_angle1 = 15  # Change this to the correct angle
aligned_sat = rotate_image(sat_image, rotation_angle1)

In [ ]:
# Rotate second image
rotation_angle2 = 15  # Change this to the correct angle
aligned_ref = rotate_image(sat_image, rotation_angle2)

In [ ]:
# Save aligned images
cv2.imwrite('aligned_satellite.png', aligned_sat)
cv2.imwrite('aligned_reference.png', aligned_ref)

Now we can do mapping and labeling

In [5]:
# Define grid size (adjust based on resolution)
grid_size = 50  # e.g., 50x50 pixels per grid cell
height, width, _ = sat_image.shape

In [6]:
# Define color-to-crop mapping (update with actual values)
color_to_crop = {
    (0, 255, 0): 'Corn',   # Example: Green represents corn
    (255, 0, 0): 'Soybean',  # Example: Red represents soybeans
    # Add more crop colors here...
}

In [7]:
# Prepare a list to store labeled data
labeled_data = []

In [8]:
# Loop through grid cells
for y in range(0, height, grid_size):
    for x in range(0, width, grid_size):
        # Extract grid section from reference image
        grid_ref = ref_image[y:y+grid_size, x:x+grid_size]
        
        # Compute the most common color in the grid
        pixels = grid_ref.reshape(-1, 3)
        unique_colors, counts = np.unique(pixels, axis=0, return_counts=True)
        dominant_color = tuple(unique_colors[np.argmax(counts)])  # Most frequent color

        # Assign crop type based on color
        crop_type = color_to_crop.get(dominant_color, 'Unknown')

        # Save grid location and crop type
        labeled_data.append([x, y, crop_type])

ValueError: attempt to get argmax of an empty sequence

In [ ]:
# Save to CSV for later use in machine learning
df = pd.DataFrame(labeled_data, columns=['X', 'Y', 'Crop_Type'])
df.to_csv('labeled_crops.csv', index=False)

print("Labeling completed and saved to labeled_crops.csv")